# Phase 5 — Multi-Sequence Validation

Replicates the Phase 5 resolution invariance finding on three additional
TUM-VI sequences to confirm the result is not specific to `room1`.

| Sequence | Scene type | Motion |
|---|---|---|
| room1 | Indoor room | Slow, handheld |
| room2 | Indoor room | Similar to room1 |
| corridor1 | Long corridor | Faster, more rotational |
| slides1 | Lecture slides | Near-planar, fast pan |

**Method:** Single-pass VGGT, 8 frames per sequence, resolutions 224–518 px.  
**Question:** Does ATE vary with resolution on sequences with different dynamics?

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

## 1 — Imports

In [ ]:
import os, sys, gc
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi            import TUMVIDataset
from src.resolution_sweep  import ResolutionSweeper, DEFAULT_RESOLUTIONS

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2 — Configure Sequences

In [ ]:
SEQUENCES    = ["room1", "room2", "corridor1", "slides1"]
N_FRAMES     = 8       # single-pass safe limit on T4 at 518 px
DOWNLOAD_DIR = "/tmp/tumvi"

print(f"Sequences: {SEQUENCES}")
print(f"Frames per sequence: {N_FRAMES}")
print(f"Resolutions: {DEFAULT_RESOLUTIONS}")

## 3 — Load Model

In [ ]:
sweeper = ResolutionSweeper(
    resolutions      = DEFAULT_RESOLUTIONS,
    conf_thresh      = 5.0,
    store_extrinsics = False,
    # chunk_size omitted → single-pass mode
)
sweeper.load_model()

## 4 — Sweep All Sequences

In [ ]:
all_dfs = {}
os.makedirs("results", exist_ok=True)

for seq in SEQUENCES:
    print(f"\n{'='*50}")
    print(f"Sequence: {seq}")
    print(f"{'='*50}")

    ds = TUMVIDataset(sequence=seq, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
    ds.download()
    data = ds.load()

    results = sweeper.run(
        image_dir     = data["image_dir"],
        max_frames    = N_FRAMES,
        gt_extrinsics = data["gt_extrinsics"],
        align         = True,
        with_scale    = True,
    )
    df = sweeper.to_dataframe(results)
    df["sequence"] = seq
    all_dfs[seq] = df

    csv_path = f"results/phase5_multisequence_{seq}.csv"
    df.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_all = pd.concat(all_dfs.values(), ignore_index=True)
df_all.to_csv("results/phase5_multisequence_all.csv", index=False)
print("\nAll sequences saved to results/phase5_multisequence_all.csv")

## 5 — ATE vs Resolution per Sequence

In [ ]:
colors = px.colors.qualitative.Plotly

fig = go.Figure()
for i, (seq, df) in enumerate(all_dfs.items()):
    fig.add_trace(go.Scatter(
        x=df["resolution"], y=df["ate_mean"],
        mode="lines+markers",
        name=seq,
        line=dict(color=colors[i], width=2),
        marker=dict(size=8),
    ))

fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")
fig.update_layout(
    title="ATE vs Resolution — Four TUM-VI Sequences (single-pass, 8 frames)",
    xaxis_title="Resolution (px)",
    yaxis_title="ATE mean (m, Umeyama-aligned)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 6 — RPE Translation vs Resolution per Sequence

In [ ]:
fig = go.Figure()
for i, (seq, df) in enumerate(all_dfs.items()):
    fig.add_trace(go.Scatter(
        x=df["resolution"], y=df["rpe_trans"],
        mode="lines+markers",
        name=seq,
        line=dict(color=colors[i], width=2),
        marker=dict(size=8),
    ))

fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")
fig.update_layout(
    title="RPE Translation vs Resolution — Four Sequences",
    xaxis_title="Resolution (px)",
    yaxis_title="RPE translation mean (m)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 7 — Rotation Error vs Resolution per Sequence
After the R_CAM_IMU fix, rotation errors should now be meaningful.

In [ ]:
fig = go.Figure()
for i, (seq, df) in enumerate(all_dfs.items()):
    fig.add_trace(go.Scatter(
        x=df["resolution"], y=df["rot_mean_deg"],
        mode="lines+markers",
        name=seq,
        line=dict(color=colors[i], width=2),
        marker=dict(size=8),
    ))

fig.add_vline(x=518, line_dash="dash", line_color="grey",
              annotation_text="Native 518 px", annotation_position="top left")
fig.update_layout(
    title="Mean Rotation Error vs Resolution — Four Sequences (post R_CAM_IMU fix)",
    xaxis_title="Resolution (px)",
    yaxis_title="Mean rotation error (deg)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 8 — Summary Table

In [ ]:
# Variance of ATE across resolutions per sequence — a flat curve has near-zero variance
summary_rows = []
for seq, df in all_dfs.items():
    ate_vals = df["ate_mean"].values
    summary_rows.append({
        "sequence":       seq,
        "ate_min":        ate_vals.min(),
        "ate_max":        ate_vals.max(),
        "ate_range":      ate_vals.max() - ate_vals.min(),
        "ate_std":        ate_vals.std(),
        "ate_224px":      df.loc[df["resolution"] == 224, "ate_mean"].values[0],
        "ate_518px":      df.loc[df["resolution"] == 518, "ate_mean"].values[0],
        "ate_pct_change": 100 * (df.loc[df["resolution"] == 224, "ate_mean"].values[0]
                                - df.loc[df["resolution"] == 518, "ate_mean"].values[0])
                               / df.loc[df["resolution"] == 518, "ate_mean"].values[0],
    })

df_summary = pd.DataFrame(summary_rows)
print("\n=== ATE sensitivity to resolution across sequences ===")
print(df_summary.to_string(index=False, float_format="{:.4f}".format))
print("\nNote: ate_range and ate_std near zero = resolution invariant.")
print("Note: ate_pct_change = (ATE_224 - ATE_518) / ATE_518 * 100.")
print("      Positive = 224px is worse; negative = 224px is better; ~0 = same.")

## 9 — Compute Cost Comparison (224 px vs 518 px)

In [ ]:
# Show time and memory savings from using 224px instead of 518px
cost_rows = []
for seq, df in all_dfs.items():
    t_224  = df.loc[df["resolution"] == 224, "time_s"].values[0]
    t_518  = df.loc[df["resolution"] == 518, "time_s"].values[0]
    mb_224 = df.loc[df["resolution"] == 224, "peak_mb"].values[0]
    mb_518 = df.loc[df["resolution"] == 518, "peak_mb"].values[0]
    cost_rows.append({
        "sequence":     seq,
        "time_224s":    t_224,
        "time_518s":    t_518,
        "speedup_x":    t_518 / t_224,
        "vram_224_mb":  mb_224,
        "vram_518_mb":  mb_518,
        "vram_saved_pct": 100 * (mb_518 - mb_224) / mb_518,
    })

df_cost = pd.DataFrame(cost_rows)
print("\n=== Compute savings: 224 px vs 518 px ===")
print(df_cost.to_string(index=False, float_format="{:.2f}".format))

avg_speedup = df_cost["speedup_x"].mean()
avg_vram    = df_cost["vram_saved_pct"].mean()
print(f"\nMean speedup across sequences : {avg_speedup:.1f}×")
print(f"Mean VRAM saved               : {avg_vram:.1f}%")